### **1. Mount Google Drive and Setup Directory Structure**

In [2]:
from google.colab import drive
import os

# Mount Google Drive to access and save files persistently
drive.mount('/content/drive')

# Define the base directory and sub-directories in Google Drive
BASE_DIR = '/content/drive/MyDrive/finrag_project'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# Create the directories if they do not exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Workspace established at: {BASE_DIR}")
print("ACTION REQUIRED: Please upload your PDF financial reports into the 'data' folder inside 'finrag_project' on your Google Drive before proceeding to Cell 4.")

Mounted at /content/drive
Workspace established at: /content/drive/MyDrive/finrag_project
ACTION REQUIRED: Please upload your PDF financial reports into the 'data' folder inside 'finrag_project' on your Google Drive before proceeding to Cell 4.


### **2. Install Required Libraries**

In [3]:
# Install libraries for PDF parsing, embedding models, and vector database integration
!pip install -qU llama-index llama-index-readers-file llama-parse
!pip install -qU llama-index-embeddings-huggingface qdrant-client llama-index-vector-stores-qdrant

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16

### **3. Configure API Keys Securely**

In [6]:
import os
from getpass import getpass

# Use getpass to securely input API credentials without hardcoding them in the notebook
print("Please enter your LlamaCloud API Key:")
os.environ["LLAMA_CLOUD_API_KEY"] = getpass()

print("Please enter your Qdrant Cluster URL (e.g., https://xxxx.qdrant.tech):")
QDRANT_URL = getpass()

print("Please enter your Qdrant API Key:")
QDRANT_API_KEY = getpass()

print("Credentials successfully loaded into environment variables.")

Please enter your LlamaCloud API Key:
··········
Please enter your Qdrant Cluster URL (e.g., https://xxxx.qdrant.tech):
··········
Please enter your Qdrant API Key:
··········
Credentials successfully loaded into environment variables.


### **4. PDF Parsing and Data Backup**

In [7]:
from llama_parse import LlamaParse
from llama_index.core import SimpleDirectoryReader

# Initialize LlamaParse prioritizing markdown format to preserve financial table structures
parser = LlamaParse(
    result_type="markdown",
    verbose=True,
    language="vi" # Optimize OCR and parsing for Vietnamese text
)

file_extractor = {".pdf": parser}

# Load all PDF documents directly from the Google Drive data directory
print(f"Loading and parsing PDFs from: {DATA_DIR}...")
documents = SimpleDirectoryReader(
    input_dir=DATA_DIR,
    file_extractor=file_extractor
).load_data()

print(f"Parsing completed! Successfully extracted {len(documents)} document objects.")

# Save the parsed markdown content to Google Drive for backup, review, and QA testing
parsed_file_path = os.path.join(OUTPUT_DIR, "parsed_documents.md")
with open(parsed_file_path, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(doc.text + "\n\n---\n\n")

print(f"Parsed markdown text has been successfully saved at: {parsed_file_path}")

/tmp/ipykernel_598/2312749508.py:1: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


Loading and parsing PDFs from: /content/drive/MyDrive/finrag_project/data...
Started parsing the file under job_id bdeae833-c799-40dd-9dc0-f9f9a05a72bb
Error while parsing the file '<bytes/buffer>': Event loop is closed
Started parsing the file under job_id 31d9eed5-948c-474b-b223-0cbe3267c4be
Error while parsing the file '<bytes/buffer>': Event loop is closed
Started parsing the file under job_id 4439f51a-4034-4c9d-9e5b-e1b5dcb0fb0f
Parsing completed! Successfully extracted 485 document objects.
Parsed markdown text has been successfully saved at: /content/drive/MyDrive/finrag_project/output/parsed_documents.md


### **5. Chunking, Embedding, and Vector Upload**

In [8]:
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from llama_index.core.node_parser import MarkdownNodeParser

# 1. Configure the Embedding Model (GPU acceleration will be utilized automatically if available)
print("Initializing HuggingFace Embedding Model (BAAI/bge-m3)...")
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
Settings.embed_model = embed_model

# 2. Chunking strategy: Use MarkdownNodeParser to prevent splitting data inside financial tables
print("Chunking documents based on markdown headers and structural elements...")
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)
print(f"Chunking completed. Generated {len(nodes)} distinct nodes (text chunks).")

# 3. Establish connection to Qdrant Cloud Vector Database
print("Connecting to Qdrant Cloud...")
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

# Initialize the vector store targeting a specific collection
collection_name = "finrag_fpt"
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name
)

# 4. Generate embeddings and push the vectors to Qdrant Cloud
print(f"Generating embeddings and uploading vectors to the '{collection_name}' collection...")
print("This may take a few minutes depending on the dataset size.")

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context,
)

print("Data Pipeline Completed Successfully! All vector representations are now stored in Qdrant Cloud.")

Initializing HuggingFace Embedding Model (BAAI/bge-m3)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Chunking documents based on markdown headers and structural elements...
Chunking completed. Generated 3845 distinct nodes (text chunks).
Connecting to Qdrant Cloud...
Generating embeddings and uploading vectors to the 'finrag_fpt' collection...
This may take a few minutes depending on the dataset size.
Data Pipeline Completed Successfully! All vector representations are now stored in Qdrant Cloud.
